# Grindhouse — Data Explorer

Reads Iceberg tables via PyIceberg + Nessie catalog.  
No Spark needed — PyIceberg reads Parquet files from MinIO directly.

**Tables available:**
- `silver.vacancies` — unified cross-source vacancies (312 rows)
- `silver.hh_vacancies` — normalized HH.ru vacancies
- `gold.hub_vacancy`, `gold.hub_company`, `gold.hub_contact`
- `gold.link_vacancy_company`, `gold.link_vacancy_contact`
- `gold.sat_vacancy_details`, `gold.sat_vacancy_source`, `gold.sat_company_details`
- `gold.sat_vacancy_score` — AI scores (311 rows, 14 priority_apply, 38 apply)

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/grindhouse')

import pandas as pd
from extractors.catalog import get_catalog, patch_table_io

catalog = get_catalog()
print('Connected to Nessie catalog')

def read_table(name: str) -> pd.DataFrame:
    """Load an Iceberg table into a pandas DataFrame."""
    tbl = patch_table_io(catalog.load_table(name))
    return tbl.scan().to_arrow().to_pandas()

print('Helper ready: read_table("silver.vacancies")')

Connected to Nessie catalog
Helper ready: read_table("silver.vacancies")


## Scoring overview

In [2]:
scores = read_table('gold.sat_vacancy_score')
print(f'Total scored: {len(scores)}')
scores['recommended_action'].value_counts()

Total scored: 311


recommended_action
skip              135
review            124
apply              38
priority_apply     14
Name: count, dtype: int64

In [3]:
# Score distribution
scores['score'].describe()

count    311.000000
mean       4.393891
std        2.310913
min        0.000000
25%        2.500000
50%        4.500000
75%        6.000000
max        9.500000
Name: score, dtype: float64

## Top vacancies (apply + priority_apply)

In [4]:
details = read_table('gold.sat_vacancy_details')
companies = read_table('gold.sat_company_details')
links = read_table('gold.link_vacancy_company')
hubs_co = read_table('gold.hub_company')

# Join company names
co_names = hubs_co[['hub_company_hk', 'company_name_bk']]
vacancy_company = links.merge(co_names, on='hub_company_hk', how='left')

top = (
    scores[scores['recommended_action'].isin(['apply', 'priority_apply'])]
    .merge(details[['hub_vacancy_hk', 'url']], on='hub_vacancy_hk', how='left')
    .merge(vacancy_company[['hub_vacancy_hk', 'company_name_bk']], on='hub_vacancy_hk', how='left')
    [['company_name_bk', 'score', 'recommended_action', 'reasoning', 'url']]
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', 80)
top

,company_name_bk,score,recommended_action,reasoning,url
0,VOXYS,9.5,priority_apply,"Аналитик данных с явным упоминанием миграции в ClickHouse, dbt-тестов, Airfl...",https://hh.ru/vacancy/131075310
1,МОДУЛЬБАНК,9.5,priority_apply,"Data Engineer junior+ с точным совпадением стека: ClickHouse, dbt, Apache Sp...",https://hh.ru/vacancy/131241409
2,НОВЕО,9.5,priority_apply,"Exact match: Python, ClickHouse, dbt, and BigQuery explicitly required — thi...",https://hh.ru/vacancy/131992098
3,YCLIENTS LLC,9.5,priority_apply,"Senior Data Engineer role explicitly mentions Trino for ad-hoc queries, Clic...",https://hh.ru/vacancy/130248999
4,FOM GROUP,9.5,priority_apply,"Data Engineer / DWH Engineer с явным требованием ClickHouse, PostgreSQL, Pyt...",https://hh.ru/vacancy/131746794
5,ООО ЛИССОМ СОЛЮШЕНС,9.0,priority_apply,"Excellent match: 3+ years DE experience, Airflow, dbt explicitly listed, col...",https://hh.ru/vacancy/131430822
6,МКК ЛУНА,9.0,priority_apply,"Data Engineer с ClickHouse, PostgreSQL, SQL оптимизацией и настройкой класте...",https://hh.ru/vacancy/131221151
7,ЛОЦИЯ,9.0,priority_apply,Data Engineer explicitly listing dbt/sqlmesh as highly desired alongside Air...,https://hh.ru/vacancy/131269491
8,ТЕХНОНИКОЛЬ,9.0,priority_apply,"Аналитик DWH с dbt, Data Vault 2.0, моделированием данных, lineage и каталог...",https://hh.ru/vacancy/129879872
9,LUCKYGROUP,9.0,priority_apply,"Data Engineer с прямым требованием ClickHouse, Postgres, Python, Pandas, Air...",https://hh.ru/vacancy/131703711


## Silver vacancies — source breakdown

In [5]:
vacancies = read_table('silver.vacancies')
print(f'Total vacancies: {len(vacancies)}')
vacancies['source'].value_counts()

Total vacancies: 312


source
hh    312
Name: count, dtype: int64

## HH salary analysis

In [6]:
hh = read_table('silver.hh_vacancies')
salary = hh[hh['salary_from'].notna()][['vacancy_name', 'salary_from', 'salary_to', 'salary_currency', 'area_name']]
print(f'Vacancies with salary: {len(salary)} / {len(hh)}')
salary.sort_values('salary_from', ascending=False).head(20)

Vacancies with salary: 45 / 312


,vacancy_name,salary_from,salary_to,salary_currency,area_name
250,Аналитик-разработчик,1100000.0,1500000.0,KZT,Алматы
306,Аналитик данных,1000000.0,1200000.0,KZT,Алматы
273,Senior/Middle+ backend Engineer,450000.0,NaN,RUR,Москва
266,Аналитик данных,400000.0,NaN,KZT,Алматы
105,Senior Data Engineer (Azure Databricks),400000.0,450000.0,RUR,Москва
189,Lead Backend Engineer / System Architect (Highload / Distributed Systems),400000.0,500000.0,RUR,Москва
98,ETL Разработчик,300000.0,320000.0,RUR,Москва
129,Разработчик BI Insight,300000.0,350000.0,RUR,Москва
249,Аналитик данных,300000.0,350000.0,RUR,Москва
158,Data Engineer,300000.0,330000.0,RUR,Москва
